# Модуль 6. Паттерны: Observer, Factory, Singleton

**Курс:** Объектно-ориентированное программирование для разработки компьютерных игр
**Модуль:** 6 — Паттерны: Observer, Factory, Singleton
**Теория:** 8 ч | **Практика:** 10 ч | **Самостоятельная работа:** 8 ч

---

## О чём этот модуль

В модуле 5 мы формализовали состояния, поведение и действия через State, Strategy и Command. Но остались три проблемы:

1. **Системы связаны напрямую.** `MovementSystem` вызывает `UI.update()` и `AudioManager.play()`. При добавлении новой системы приходится править все существующие.
2. **Создание объектов зашито в код.** `Game` сам делает `Enemy("гоблин", hp=50)` — новые типы врагов требуют правки `Game`.
3. **Есть объекты, которые должны существовать в единственном экземпляре.** `AudioManager` и `Config` создаются по несколько раз, из-за чего настройки «расползаются».

Три паттерна решают эти проблемы:

- **Observer** — объекты подписываются на события вместо прямых вызовов. Слабая связанность.
- **Factory** — создание объектов выносится в отдельный класс. Новые типы объектов без правки клиента.
- **Singleton** — гарантирует единственный экземпляр и глобальную точку доступа. С осторожностью.

## Где мы находимся в проекте

После модуля 5 проект «Тайна исчезнувшего дива» имеет:

| Слой | Что есть |
|---|---|
| **Абстракции** | `GameObject` (ABC), протоколы способностей |
| **Сущности** | `Detective`, `Div`, `Location`, `Clue`, `Entity` + компоненты |
| **Системы** | Командные (`InvestigationSystem`, ...) и обновления (`MovementSystem`, ...) |
| **Состояния и поведение** | `GameState` FSM, `BehaviorStrategy`, `CommandHistory` |
| **Оркестрация** | `Game`, `GameLoop`, `WorldState` |
| **Инфраструктура** | `UI`, `InputHandler`, `Renderer` |

**Чего не хватает:** системы связаны напрямую, создание объектов зашито в `Game`, общие ресурсы не управляются.

## Что вы изучите

1. Паттерн Observer: события и подписчики.
2. EventBus (Pub-Sub) как обобщение Observer.
3. Подводные камни Observer: утечки, порядок, исключения.
4. Паттерн Factory и его виды: Simple Factory, Factory Method, Abstract Factory.
5. Фабрики врагов и предметов в проекте.
6. Паттерн Singleton и почему его применяют осторожно.
7. Потокобезопасная реализация Singleton в Python.
8. Интеграция трёх паттернов в проект.

## Содержание
1. Паттерн Observer: события вместо вызовов
2. EventBus: шина событий
3. Подводные камни Observer
4. Паттерн Factory: управление созданием
5. Виды фабрик
6. Паттерн Singleton
7. Интеграция в проект
8. Итоги и литература

In [ ]:
import sys
print(f"Python: {sys.version}")
assert sys.version_info >= (3, 10), "Требуется Python 3.10+"

---
# 1. Паттерн Observer: события вместо вызовов

## 1.1. Проблема сильной связанности

Представьте: в игре есть достижения, UI, звук и аналитика. Когда игрок находит улику, нужно:

- показать уведомление (UI);
- проиграть звук (AudioManager);
- отметить достижение (Achievements);
- записать в лог (Analytics).

**Наивная реализация** — `InvestigationSystem` напрямую вызывает каждый сервис:

```python
class InvestigationSystem:
    def on_clue_found(self, clue):
        self.ui.notify(clue)
        self.audio.play("found")
        self.achievements.unlock("first_clue")
        self.analytics.log("clue_found", clue)
```

**Проблемы:**
- `InvestigationSystem` знает про UI, звук, достижения, аналитику — нарушение SRP и DIP.
- Каждый новый потребитель события (например, подсказка Кузи) — правка `on_clue_found` (нарушение OCP).
- Системы нельзя переиспользовать: они прибиты друг к другу.

## 1.2. Идея Observer

**Observer** инвертирует связь: субъект не знает о подписчиках, он лишь **публикует события**. Подписчики **сами** решают, подписываться или нет.

| Роль | Что делает | Пример |
|---|---|---|
| **Subject (издатель)** | Хранит подписчиков, публикует события | `GameEvents` |
| **Observer (подписчик)** | Реагирует на события | `AchievementSystem`, `AudioManager` |
| **Event (событие)** | Данные о произошедшем | `ClueFound(clue)` |

## 1.3. Ключевая идея

**Издатель не знает подписчиков.** Он вызывает общий метод `update(event)` у всех, кто подписался. Подписчики сами решают, как реагировать.

## 1.4. Как это улучшит проект

**До:** `InvestigationSystem.on_clue_found` дёргает UI, звук, достижения напрямую.

**После:** `InvestigationSystem` публикует `ClueFound`. UI, звук, достижения — подписчики. Новая система (например, подсказки Кузи) просто подписывается, не трогая существующий код.

## Для углублённого изучения

- Nystrom R. «Game Programming Patterns»: [Observer](https://gameprogrammingpatterns.com/observer.html) — каноническое объяснение.
- [Refactoring Guru: Observer](https://refactoring.guru/design-patterns/observer).
- Документация Python: [слабые ссылки weakref](https://docs.python.org/3/library/weakref.html).

---
# 2. EventBus: шина событий

## 2.1. От Observer к Pub-Sub

**EventBus** (шина событий) — это обобщение Observer: единая точка, через которую проходят все события игры. Издатели публикуют в шину, подписчики подписываются на типы событий.

Отличия:
- **Observer** — издатель сам хранит список подписчиков.
- **Pub-Sub (EventBus)** — шина отделяет издателя от подписчиков, они не знают друг о друге.

## 2.2. Простая реализация EventBus

```python
class EventBus:
    def __init__(self):
        self._subscribers = {}  # type -> [handler]

    def subscribe(self, event_type, handler):
        self._subscribers.setdefault(event_type, []).append(handler)

    def unsubscribe(self, event_type, handler):
        handlers = self._subscribers.get(event_type, [])
        if handler in handlers:
            handlers.remove(handler)

    def publish(self, event):
        for handler in self._subscribers.get(type(event), []):
            handler(event)
```

## 2.3. События как dataclass

События удобно описывать как `@dataclass` — они иммутабельны и самодокументированы:

```python
@dataclass(frozen=True)
class ClueFound:
    clue: Clue
    location: str

@dataclass(frozen=True)
class EnemyDefeated:
    enemy_type: str
    position: tuple[float, float]
```

## 2.4. Как это улучшит проект

Единая `EventBus` в `Game`:
- `InvestigationSystem` публикует `ClueFound`.
- `AchievementSystem`, `AudioManager`, `UI` — подписчики.
- Новые системы подписываются, не трогая издателя.

## Для углублённого изучения

- Nystrom R. «Game Programming Patterns»: [Observer](https://gameprogrammingpatterns.com/observer.html), раздел про Event Queue.
- [Refactoring Guru: Observer](https://refactoring.guru/design-patterns/observer).

---
# 3. Подводные камни Observer

Observer — мощный, но не бесплатный. Четыре типичные проблемы.

## 3.1. Утечки памяти

Если подписчик подписался и не отписался, шина держит на него сильную ссылку — объект не удалится сборщиком мусора.

**Решения:**
- явный `unsubscribe` при удалении объекта;
- слабые ссылки (`weakref.WeakMethod`, `weakref.ref`);
- подписка на время жизни сцены (отписка при выходе из состояния).

## 3.2. Порядок обработки

Если достижение должно сработать **до** звука, порядок подписки важен. По умолчанию порядок = порядок подписки, но это неявно и хрупко.

**Решения:**
- приоритеты подписчиков;
- отдельные шины для разных доменов (UI-события, игровые события).

## 3.3. Исключения в подписчиках

Если один подписчик бросил исключение, остальные не получат событие.

**Решение:** оборачивать вызов каждого обработчика в `try/except` и логировать ошибку, не прерывая цепочку.

## 3.4. Синхронность

По умолчанию Observer синхронен: `publish` блокирует, пока все подписчики не отработают. Для тяжёлых обработчиков это проблема (подтормаживания кадра).

**Решения:**
- очередь событий и обработка в конце кадра;
- отдельный поток (аккуратно — см. модуль 7).

## Как это улучшит проект

В «Тайне исчезнувшего дива» мы заложим явный `unsubscribe` и `try/except` в `EventBus`, чтобы избежать утечек и падений цепочки.

## Для углублённого изучения

- Nystrom R. «Game Programming Patterns»: [Observer](https://gameprogrammingpatterns.com/observer.html), раздел про подводные камни.
- Документация Python: [weakref](https://docs.python.org/3/library/weakref.html).

---
# 4. Паттерн Factory: управление созданием

## 4.1. Проблема: создание объектов зашито в код

```python
class Game:
    def spawn_enemy(self, kind: str):
        if kind == "гоблин":
            return Enemy("гоблин", hp=50, speed=30)
        elif kind == "волк":
            return Enemy("волк", hp=80, speed=60)
        elif kind == "демон":
            return Enemy("демон", hp=200, speed=40)
        # ... ещё 10 elif
```

**Проблемы:**
- Каждый новый тип врага — новый `elif` (нарушение OCP).
- Логика создания размазана по `Game` (нарушение SRP).
- Невозможно переиспользовать конфигурацию врагов.

## 4.2. Идея Factory

**Factory** — это объект, который **инкапсулирует создание** других объектов. Клиент просит «дай врага типа X», а фабрика знает, как его собрать.

| Роль | Что делает | Пример |
|---|---|---|
| **Product** | Что создаём | `Enemy`, `Item` |
| **Factory** | Знает, как создавать | `EnemyFactory` |
| **Client** | Просит создать | `Game` |

## 4.3. Как это улучшит проект

**До:** `Game.spawn_enemy` с `if/elif`.

**После:** `enemy_factory.create("гоблин")`. Новый тип врага = новая запись в фабрике, `Game` не меняется.

## Для углублённого изучения

- Nystrom R. «Game Programming Patterns»: [Prototype](https://gameprogrammingpatterns.com/prototype.html) — родственный паттерн (создание через клонирование).
- [Refactoring Guru: Factory Method](https://refactoring.guru/design-patterns/factory-method).

---
# 5. Виды фабрик

## 5.1. Simple Factory (простая фабрика)

Один класс с методом, который по параметру создаёт нужный объект. Не является «настоящим» паттерном GoF, но самый частый на практике.

```python
class EnemyFactory:
    def create(self, kind: str) -> Enemy:
        return ENEMY_SPECS[kind].make()
```

Плюс: просто. Минус: метод растёт, но это уже лучше, чем `if/elif` в `Game`.

## 5.2. Factory Method (фабричный метод)

Абстрактный метод создания переопределяется подклассами. Каждый подкласс фабрики создаёт свой тип.

```python
class EnemyFactory(ABC):
    @abstractmethod
    def create(self) -> Enemy: ...

class GoblinFactory(EnemyFactory):
    def create(self): return Enemy("гоблин", hp=50)

class DemonFactory(EnemyFactory):
    def create(self): return Enemy("демон", hp=200)
```

Плюс: расширение через новые подклассы, а не правку. Минус: много классов.

## 5.3. Abstract Factory (абстрактная фабрика)

Фабрика фабрик: создаёт **семейства** связанных объектов. Например, фабрика «лесной биом» создаёт волков и лесные предметы, фабрика «подземелье» — демонов и реликвии.

```python
class BiomeFactory(ABC):
    @abstractmethod
    def create_enemy(self) -> Enemy: ...
    @abstractmethod
    def create_item(self) -> Item: ...

class ForestBiome(BiomeFactory):
    def create_enemy(self): return Enemy("волк", hp=80)
    def create_item(self): return Item("гриб")
```

## 5.4. Сравнение

| Вид | Когда | Расширение |
|---|---|---|
| Simple Factory | Один тип, параметр | Правка метода |
| Factory Method | Один тип, много подклассов | Новый подкласс |
| Abstract Factory | Семейства связанных объектов | Новая фабрика |

## 5.5. Фабрика с конфигурацией (dataclass)

В проекте удобно хранить характеристики в `dataclass` и создавать объекты по данным:

```python
@dataclass(frozen=True)
class EnemySpec:
    kind: str
    hp: int
    speed: int
    damage: int

class EnemyFactory:
    def __init__(self, specs: dict[str, EnemySpec]):
        self._specs = specs
    def create(self, kind: str) -> Enemy:
        spec = self._specs[kind]
        return Enemy(spec.kind, spec.hp, spec.speed, spec.damage)
```

## Для углублённого изучения

- [Refactoring Guru: Abstract Factory](https://refactoring.guru/design-patterns/abstract-factory), [Factory Method](https://refactoring.guru/design-patterns/factory-method).
- Gamma E. et al. «Design Patterns», главы про Factory Method и Abstract Factory.

---
# 6. Паттерн Singleton

## 6.1. Что такое Singleton

**Singleton** гарантирует, что у класса есть **ровно один экземпляр**, и предоставляет глобальную точку доступа к нему.

## 6.2. Зачем в играх

Некоторые ресурсы логично держать в одном экземпляре:
- **AudioManager** — один звуковой движок, одна громкость;
- **Config** — одна конфигурация на всю игру;
- **InputManager**, **AchievementManager** — единая точка правды.

Если таких объектов несколько — настройки «расползаются», звук двоится.

## 6.3. Реализация в Python

```python
class AudioManager:
    _instance = None

    def __new__(cls):
        if cls._instance is None:
            cls._instance = super().__new__(cls)
            cls._instance._init_once()
        return cls._instance
```

## 6.4. Почему Singleton применяют осторожно

Singleton критикуют как «анти-паттерн» по причинам:
- **Глобальное состояние** — неявные зависимости, труднее тестировать.
- **Скрытая связанность** — любой код может дотянуться до синглтона.
- **Сложно подменять** в тестах (нужен сброс `_instance`).
- **Нарушает SRP/DIP** — класс управляет и логикой, и своим созданием.

**Правило:** используйте Singleton только там, где единственность **требуется доменом** (звук, конфиг), а не для удобства доступа.

## 6.5. Потокобезопасность

`__new__` с проверкой `if _instance is None` **не потокобезопасен**: два потока могут одновременно пройти проверку и создать два экземпляра.

**Решение** — блокировка:

```python
import threading

class AudioManager:
    _instance = None
    _lock = threading.Lock()

    def __new__(cls, *args, **kwargs):
        if cls._instance is None:
            with cls._lock:
                if cls._instance is None:
                    cls._instance = super().__new__(cls)
        return cls._instance
```

Двойная проверка (double-checked locking) с `Lock` защищает от гонки.

## 6.6. Альтернативы Singleton

Вместо Singleton часто лучше **внедрение зависимостей**: создать один `AudioManager` в `main.py` и передать в конструкторы. Это тестируемее и явнее.

## Для углублённого изучения

- [Refactoring Guru: Singleton](https://refactoring.guru/design-patterns/singleton).
- Документация Python: [threading](https://docs.python.org/3/library/threading.html).
- Martin R. C. «Clean Architecture», глава про границы и управление зависимостями.

---
# 7. Интеграция в проект

## 7.1. Как паттерны сочетаются

| Паттерн | Где | Что улучшит |
|---|---|---|
| **Observer** | `EventBus` | Системы слабо связаны |
| **Factory** | `EnemyFactory`, `ItemFactory` | Создание объектов расширяемо |
| **Singleton** | `AudioManager`, `Config` | Единственный экземпляр ресурсов |

## 7.2. Пример взаимодействия

```
Game
├── event_bus: EventBus (Observer)
│   ├── AchievementSystem (подписчик)
│   ├── AudioManager (подписчик)
│   └── UI (подписчик)
├── enemy_factory: EnemyFactory (Factory)
├── item_factory: ItemFactory (Factory)
└── audio: AudioManager (Singleton)
```

## 7.3. Что появится в проекте

- `EventBus` с подпиской/отпиской и обработкой исключений.
- `EnemyFactory` и `ItemFactory` на базе `dataclass`-конфигураций.
- `AudioManager` и `Config` как потокобезопасные Singleton.
- События `ClueFound`, `EnemyDefeated`, `ItemPickedUp`.

## 7.4. Как это подготовит модуль 7

- Observer подготовит переход к слоистой архитектуре: домен публикует события, presentation подписывается.
- Factory и Singleton станут «инфраструктурой», которую в модуле 7 мы вынесем в отдельный слой.
- EventBus пригодится для связи потоков и асинхронных задач.

## Литература

**Базовая:**
- Nystrom R. «Game Programming Patterns»: [Observer](https://gameprogrammingpatterns.com/observer.html).
- Gamma E. et al. «Design Patterns» (GoF), главы про Observer, Factory Method, Abstract Factory, Singleton.

**Углублённая:**
- [Refactoring Guru: Observer](https://refactoring.guru/design-patterns/observer), [Factory Method](https://refactoring.guru/design-patterns/factory-method), [Abstract Factory](https://refactoring.guru/design-patterns/abstract-factory), [Singleton](https://refactoring.guru/design-patterns/singleton).
- Документация Python: [weakref](https://docs.python.org/3/library/weakref.html), [threading](https://docs.python.org/3/library/threading.html).

## Что дальше

В практической работе вы добавите в «Тайну исчезнувшего дива»:
- EventBus через Observer;
- фабрики врагов и предметов;
- Singleton для AudioManager и Config.